# Stream a synthetic scholarly corpus with bounded records

**Audience:** Python developers preparing reproducible data workflows. **Time:** about 15–25 minutes. **Execution:** offline; no credentials, paid APIs, or downloaded datasets.

Read JSON Lines incrementally, count malformed records and retain bounded diagnostic samples with line numbers, validate citation links, and aggregate without loading an entire corpus.

## Prerequisites and setup

Use the tested Python environment and exact dependency versions in the [collection setup](../../../README.md). Install from the collection root before opening this notebook, then select that environment's kernel. This notebook uses Python standard library and pandas. Restart the kernel and run all cells in order. No working-directory assumptions or private files are required.

## Data provenance

All examples below are newly authored synthetic fixtures. Names, identifiers, coordinates and values are invented for this tutorial and do not represent real people, accounts, organizations or operational records. Code is newly authored for this example; the references explain the underlying APIs.

Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Create a deliberately imperfect stream

JSON Lines stores one JSON object per line. It can be read incrementally, unlike a naive `json.load` of a huge top-level array. This small stream includes malformed JSON and a schema error so the quality report is observable.

In [ ]:
import io
import json
from collections import Counter

import pandas as pd

records = [
    {"id": "paper-1", "title": "Synthetic storage study", "year": 2022, "references": []},
    {
        "id": "paper-2",
        "title": "Synthetic retrieval study",
        "year": 2023,
        "references": ["paper-1"],
    },
    {
        "id": "paper-3",
        "title": "Synthetic indexing study",
        "year": 2023,
        "references": ["paper-1", "paper-2"],
    },
]
text = (
    "\n".join(
        [
            json.dumps(records[0]),
            "{broken",
            json.dumps(records[1]),
            json.dumps({"id": "bad", "year": "unknown"}),
            json.dumps(records[2]),
        ]
    )
    + "\n"
)

## 2. Bound reads before decoding

Use `readline(limit + 1)` to detect an oversized record before parsing it. An oversized line stops ingestion: attempting to quarantine its complete contents would undermine the memory bound. Malformed small records are counted and skipped with explicit reasons, never silently discarded.

The rejection log keeps exact totals for two fixed reason categories and at most five line-number samples. It never retains raw rejected content. This prevents an all-invalid corpus from growing an unbounded error list. Year counts are also bounded by the schema's 1900–2100 range. A production quarantine should stream rejected records into separately protected storage rather than accumulate them in memory.

In [ ]:
class RejectionLog:
    def __init__(self, sample_limit=5):
        if type(sample_limit) is not int or sample_limit < 0:
            raise ValueError("Sample limit must be a nonnegative integer")
        self.sample_limit = sample_limit
        self.counts = Counter()
        self.samples = []

    def record(self, line, reason):
        if reason not in {"invalid JSON", "schema mismatch"}:
            raise ValueError("Unknown rejection category")
        self.counts[reason] += 1
        if len(self.samples) < self.sample_limit:
            self.samples.append({"line": line, "reason": reason})

    @property
    def total(self):
        return sum(self.counts.values())


def stream_papers(handle, rejected, max_line_chars=4096):
    if max_line_chars < 1:
        raise ValueError("Line limit must be positive")
    line_number = 0
    while True:
        line = handle.readline(max_line_chars + 1)
        if not line:
            break
        line_number += 1
        if len(line) > max_line_chars:
            raise ValueError(f"Line {line_number} exceeds record size limit")
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            rejected.record(line_number, "invalid JSON")
            continue
        valid = (
            isinstance(row, dict)
            and isinstance(row.get("id"), str)
            and bool(row["id"])
            and isinstance(row.get("title"), str)
            and bool(row["title"].strip())
            and type(row.get("year")) is int
            and 1900 <= row["year"] <= 2100
            and isinstance(row.get("references"), list)
            and all(isinstance(ref, str) and ref for ref in row["references"])
        )
        if not valid:
            rejected.record(line_number, "schema mismatch")
            continue
        yield row


rejected = RejectionLog()
year_counts = Counter(row["year"] for row in stream_papers(io.StringIO(text), rejected))
assert year_counts == {2022: 1, 2023: 2}
assert [row["line"] for row in rejected.samples] == [2, 4]
assert rejected.total == 2
pd.DataFrame(rejected.samples)

## 3. Compute citation counts on the accepted subset

This second pass materializes only the tiny fixture to demonstrate referential checks. A production corpus requires a disk-backed index for global IDs, duplicate detection and unresolved references. Year aggregation above is streaming; this graph step is explicitly not constant-memory.

In [ ]:
accepted = list(stream_papers(io.StringIO(text), RejectionLog(sample_limit=0)))
identifiers = {row["id"] for row in accepted}
assert len(identifiers) == len(accepted), "Duplicate article identifiers"
inbound = Counter()
unknown_references = []
for row in accepted:
    for target in set(row["references"]):
        if target == row["id"]:
            raise ValueError("Self citation in this teaching schema")
        if target in identifiers:
            inbound[target] += 1
        else:
            unknown_references.append((row["id"], target))
assert not unknown_references
assert inbound["paper-1"] == 2
pd.DataFrame([{**row, "inbound_in_subset": inbound[row["id"]]} for row in accepted])

## 4. Verify empty, oversized and heavily rejected inputs

Record limits here are measured in decoded characters. For compressed input, also limit expanded bytes and decompression ratios before handing data to this parser.

In [ ]:
assert list(stream_papers(io.StringIO(""), RejectionLog())) == []
try:
    list(stream_papers(io.StringIO("x" * 100), RejectionLog(), max_line_chars=20))
except ValueError:
    print("Oversized record stopped before JSON parsing.")
else:
    raise AssertionError("Record size limit not enforced")
print(
    {"accepted": sum(year_counts.values()), "rejected": rejected.total, "years": dict(year_counts)}
)

many_rejections = RejectionLog(sample_limit=3)
assert list(stream_papers(io.StringIO("{bad\n" * 1000), many_rejections)) == []
assert many_rejections.total == 1000
assert many_rejections.counts == {"invalid JSON": 1000}
assert len(many_rejections.samples) == 3
assert [sample["line"] for sample in many_rejections.samples] == [1, 2, 3]
print(
    {
        "rejection_stress_total": many_rejections.total,
        "retained_diagnostic_samples": len(many_rejections.samples),
    }
)

## Validation and interpretation

The assertions above are executable acceptance criteria. A failed assertion is a failed example: inspect the inputs and kernel versions before continuing. Inspect the displayed tables or figures as well; successful execution alone does not establish fitness for a different dataset.

## Limitations and extensions

The fixture is not a real bibliographic dataset and supports no scientific conclusions. For a top-level JSON array use a tested incremental parser instead of pretending JSON Lines accepts it. Real corpus ingestion needs encoding policy, persistent quarantine, duplicate detection, restart checkpoints and dataset-license review.

## Cleanup

This notebook creates no persistent files, network sessions or cloud resources. Restart the kernel to discard in-memory objects. If you adapt it to real data, keep inputs outside a public repository and review every output before sharing.

## References and attribution

- [Python JSON decoder behavior](https://docs.python.org/3/library/json.html)
- [Python text I/O](https://docs.python.org/3/library/io.html)

The example is prepared for editorial review; it is not an IBM endorsement or a claim of production certification.